# Project - Airline AI Assistant (Ollama local + HuggingFace + Gradio Audio)

### Stack :
- Chat          : Ollama local → llama3.2:3b
- Function call : Ollama tools API (format OpenAI)
- Images        : Hugging Face Inference → FLUX.1-schnell (gratuit, sans CB)
- TTS           : gTTS → audio affiché directement dans Gradio

In [11]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import tempfile
import requests
from io import BytesIO
from dotenv import load_dotenv
import gradio as gr
from PIL import Image
from gtts import gTTS
from ollama import Client

In [12]:
# ── Initialisation ────────────────────────────────────────────────────────────
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    print(f"HF Token trouvé, commence par : {HF_TOKEN[:8]}")
else:
    print("⚠️  HF_TOKEN non défini dans .env")

ollama_client = Client(host="http://localhost:11434")
MODEL       = "llama3.2:3b"
IMAGE_MODEL = "black-forest-labs/FLUX.1-schnell"

# System Prompt
system_message = (
    "You are a helpful assistant for an Airline called FlightAI. "
    "Give short, courteous answers, no more than 1 sentence. "
    "Always be accurate. If you don't know the answer, say so."
)

HF Token trouvé, commence par : hf_AZDlP


In [13]:
# ── Données & outil métier ────────────────────────────────────────────────────
ticket_prices = {
    "london": "$799",
    "paris":  "$899",
    "tokyo":  "$1400",
    "berlin": "$499",
}

def get_ticket_price(destination_city: str) -> str:
    print(f"Tool get_ticket_price called for {destination_city}")
    return ticket_prices.get(destination_city.lower(), "Unknown")

In [14]:
# ── Déclaration de l'outil pour Ollama ───────────────────────────────────────
# Ollama supporte le format OpenAI tools nativement
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": (
                "Get the price of a return ticket to the destination city. "
                "Call this whenever you need to know the ticket price, for example "
                "when a customer asks 'How much is a ticket to this city'."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "destination_city": {
                        "type": "string",
                        "description": "The city that the customer wants to travel to",
                    }
                },
                "required": ["destination_city"],
            },
        },
    }
]

In [15]:
# ── Dispatch des tool calls ───────────────────────────────────────────────────
AVAILABLE_TOOLS = {"get_ticket_price": get_ticket_price}

def handle_tool_calls(tool_calls: list) -> tuple[list[dict], str | None]:
    """
    Exécute tous les tool_calls retournés par Ollama.
    Retourne (liste de messages tool_result, ville détectée ou None).
    """
    results = []
    city_found = None

    for tc in tool_calls:
        fn_name = tc.function.name
        fn_args = tc.function.arguments  # dict

        if fn_name in AVAILABLE_TOOLS:
            result = AVAILABLE_TOOLS[fn_name](**fn_args)
            if fn_name == "get_ticket_price":
                city_found = fn_args.get("destination_city")
        else:
            result = f"Unknown tool: {fn_name}"

        results.append({
            "role": "tool",
            "content": str(result),
        })

    return results, city_found

In [22]:
# ── Génération d'image via Hugging Face Inference ─────────────────────────────
def artist(city: str) -> Image.Image | None:
    """
    Utilise FLUX.1-schnell via l'API Inference de HuggingFace.
    Gratuit avec un compte HF (pas de CB requise).
    """
    api_url = f"https://router.huggingface.co/hf-inference/models/{IMAGE_MODEL}"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {
        "inputs": (
            f"A vacation in {city}, showing tourist spots and everything unique "
            f"about {city}, vibrant pop-art style"
        )
    }

    print(f"Generating image for {city} via HuggingFace ({IMAGE_MODEL})...")
    response = requests.post(api_url, headers=headers, json=payload, timeout=60)

    if response.status_code == 200:
        return Image.open(BytesIO(response.content))
    else:
        print(f"HF image error {response.status_code}: {response.text[:200]}")
        return None

In [23]:
# ── Text-to-Speech → fichier temporaire pour Gradio ──────────────────────────
def talker(message: str) -> str:
    """
    Génère un fichier MP3 via gTTS et retourne son chemin.
    Gradio lit directement le chemin pour l'afficher dans gr.Audio.
    """
    tts = gTTS(text=message, lang="en")
    tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
    tts.save(tmp.name)
    return tmp.name  # ← chemin retourné à gr.Audio

In [24]:
# ── Normalisation contenu Gradio 6.x ─────────────────────────────────────────
def extract_text(raw) -> str:
    if isinstance(raw, list):
        return " ".join(
            p["text"] for p in raw if isinstance(p, dict) and "text" in p
        )
    return raw or ""

In [25]:
def chat(history):
    """
    Retourne (history, image_ou_None, chemin_audio_ou_None)
    """
    messages = [{"role": "system", "content": system_message}]
    for msg in history:
        messages.append({
            "role":    msg["role"],
            "content": extract_text(msg["content"]),
        })

    # ── Premier appel Ollama ──────────────────────────────────────────────────
    response      = ollama_client.chat(model=MODEL, messages=messages, tools=tools)
    assistant_msg = response.message
    image         = None

    # ── Gestion tool calls ────────────────────────────────────────────────────
    if assistant_msg.tool_calls:
        messages.append({
            "role":       "assistant",
            "content":    assistant_msg.content or "",
            "tool_calls": assistant_msg.tool_calls,
        })

        tool_results, city = handle_tool_calls(assistant_msg.tool_calls)
        messages.extend(tool_results)

        if city:
            try:
                image = artist(city)
            except Exception as e:
                print(f"Image generation failed: {e}")

        # Second appel pour la réponse finale
        response      = ollama_client.chat(model=MODEL, messages=messages, tools=tools)
        assistant_msg = response.message

    # ── Réponse finale ────────────────────────────────────────────────────────
    reply   = assistant_msg.content or "Sorry, I couldn't generate a response."
    history = history + [{"role": "assistant", "content": reply}]

    # TTS → chemin fichier pour gr.Audio
    audio_path = None
    try:
        audio_path = talker(reply)
    except Exception as e:
        print(f"TTS failed: {e}")

    return history, image, audio_path

In [ ]:
# ── Interface Gradio ──────────────────────────────────────────────────────────
with gr.Blocks(title="FlightAI Assistant") as ui:
    with gr.Row():
        chatbot      = gr.Chatbot(height=500)
        image_output = gr.Image(height=500)
    with gr.Row():
        # gr.Audio avec autoplay=True → la réponse se joue automatiquement
        audio_output = gr.Audio(
            label="Assistant voice",
            autoplay=True,
            visible=True,
        )
    with gr.Row():
        entry = gr.Textbox(label="Chat with our FlightAI Assistant (Ollama local):")
    with gr.Row():
        clear = gr.Button("Clear")

    def do_entry(message, history):
        history = history + [{"role": "user", "content": message}]
        return "", history

    entry.submit(
        do_entry,
        inputs=[entry, chatbot],
        outputs=[entry, chatbot],
    ).then(
        chat,
        inputs=chatbot,
        outputs=[chatbot, image_output, audio_output],
    )

ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://1d161867348909178e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Tool get_ticket_price called for Tokyo
Generating image for Tokyo via HuggingFace (black-forest-labs/FLUX.1-schnell)...
Tool get_ticket_price called for London
Generating image for London via HuggingFace (black-forest-labs/FLUX.1-schnell)...
